# Softmax回归与图像分类

本notebook介绍分类问题和Softmax回归:
- Fashion-MNIST图像分类数据集
- Softmax回归的原理
- 从零开始实现Softmax回归
- 使用PyTorch的简洁实现

这是深度学习中第一个真正的分类任务!

## 第一部分: Fashion-MNIST数据集

### 1.1 数据集简介

**Fashion-MNIST**是一个服饰图像数据集:
- **10个类别**: T恤、裤子、套衫、连衣裙、外套、凉鞋、衬衫、运动鞋、包、短靴
- **训练集**: 60,000张图像
- **测试集**: 10,000张图像
- **图像尺寸**: 28×28像素,灰度图(单通道)

相比MNIST手写数字数据集,Fashion-MNIST更具挑战性,更接近实际应用。

In [ ]:
import torch
import torchvision
from torch.utils import data
from torchvision import transforms
import matplotlib.pyplot as plt

### 1.2 下载和加载数据集

In [ ]:
# ToTensor将图像转换为张量,并归一化到[0,1]
trans = transforms.ToTensor()

# 下载训练集和测试集
mnist_train = torchvision.datasets.FashionMNIST(
    root="../data", train=True, transform=trans, download=True)
mnist_test = torchvision.datasets.FashionMNIST(
    root="../data", train=False, transform=trans, download=True)

print(f'训练集大小: {len(mnist_train)}')
print(f'测试集大小: {len(mnist_test)}')
print(f'图像形状: {mnist_train[0][0].shape}')

### 1.3 可视化数据

In [ ]:
def get_fashion_mnist_labels(labels):
    """返回Fashion-MNIST数据集的文本标签"""
    text_labels = ['T恤', '裤子', '套衫', '连衣裙', '外套',
                   '凉鞋', '衬衫', '运动鞋', '包', '短靴']
    return [text_labels[int(i)] for i in labels]

def show_images(imgs, num_rows, num_cols, titles=None, scale=1.5):
    """绘制图像列表"""
    figsize = (num_cols * scale, num_rows * scale)
    _, axes = plt.subplots(num_rows, num_cols, figsize=figsize)
    axes = axes.flatten()
    for i, (ax, img) in enumerate(zip(axes, imgs)):
        ax.imshow(img.squeeze().numpy(), cmap='gray')
        ax.axes.get_xaxis().set_visible(False)
        ax.axes.get_yaxis().set_visible(False)
        if titles:
            ax.set_title(titles[i], fontproperties='SimHei')
    plt.tight_layout()
    return axes

In [ ]:
# 显示前18个样本
X, y = next(iter(data.DataLoader(mnist_train, batch_size=18)))
show_images(X, 2, 9, titles=get_fashion_mnist_labels(y))
plt.show()

### 1.4 创建数据迭代器

In [ ]:
batch_size = 256

# 创建数据加载器
train_iter = data.DataLoader(mnist_train, batch_size, shuffle=True, num_workers=4)
test_iter = data.DataLoader(mnist_test, batch_size, shuffle=False, num_workers=4)

print(f'每个批次的形状:')
for X, y in train_iter:
    print(f'  X: {X.shape}, dtype: {X.dtype}')
    print(f'  y: {y.shape}, dtype: {y.dtype}')
    break

---

## 第二部分: Softmax回归理论

### 2.1 从回归到分类

**回归 vs 分类**:
- **回归**: 预测连续值(如房价)
- **分类**: 预测离散类别(如图像类别)

### 2.2 独热编码(One-Hot Encoding)

将类别表示为向量,只有对应类别的位置为1,其余为0:
- 类别0: [1, 0, 0, ..., 0]
- 类别1: [0, 1, 0, ..., 0]
- 类别9: [0, 0, 0, ..., 1]

### 2.3 Softmax回归模型

**网络结构**: 输入 → 全连接层 → Softmax → 输出概率

**数学表达**:
1. 线性变换: $\mathbf{o} = \mathbf{W}\mathbf{x} + \mathbf{b}$
2. Softmax函数: $\hat{y}_j = \frac{\exp(o_j)}{\sum_k \exp(o_k)}$

**Softmax的性质**:
- 所有输出都在(0, 1)之间
- 所有输出之和为1
- 可以解释为概率分布

### 2.4 交叉熵损失

$$L = -\sum_{j=1}^{q} y_j \log \hat{y}_j$$

其中$y$是真实标签(独热编码),$\hat{y}$是预测概率。

---

## 第三部分: 从零开始实现Softmax回归

In [ ]:
import torch
from IPython import display

### 3.1 初始化模型参数

In [ ]:
num_inputs = 784  # 28 * 28
num_outputs = 10  # 10个类别

W = torch.normal(0, 0.01, size=(num_inputs, num_outputs), requires_grad=True)
b = torch.zeros(num_outputs, requires_grad=True)

print(f'权重W形状: {W.shape}')
print(f'偏置b形状: {b.shape}')

### 3.2 定义Softmax函数

In [ ]:
def softmax(X):
    """Softmax函数"""
    X_exp = torch.exp(X)
    partition = X_exp.sum(1, keepdim=True)
    return X_exp / partition  # 广播机制

# 测试softmax
X_test = torch.normal(0, 1, (2, 5))
X_prob = softmax(X_test)
print('输入:')
print(X_test)
print('\nSoftmax输出(概率):')
print(X_prob)
print('\n每行的和:', X_prob.sum(1))

### 3.3 定义模型

In [ ]:
def net(X):
    """Softmax回归模型"""
    return softmax(torch.matmul(X.reshape((-1, W.shape[0])), W) + b)

### 3.4 定义交叉熵损失函数

In [ ]:
def cross_entropy(y_hat, y):
    """交叉熵损失函数"""
    return -torch.log(y_hat[range(len(y_hat)), y])

# 测试
y_test = torch.tensor([0, 2])
y_hat_test = torch.tensor([[0.1, 0.3, 0.6], [0.3, 0.2, 0.5]])
print('交叉熵损失:', cross_entropy(y_hat_test, y_test))

### 3.5 计算分类准确率

In [ ]:
def accuracy(y_hat, y):
    """计算预测正确的数量"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())

# 测试
print('准确预测数量:', accuracy(y_hat_test, y_test))
print('准确率:', accuracy(y_hat_test, y_test) / len(y_test))

### 3.6 评估模型准确率

In [ ]:
def evaluate_accuracy(net, data_iter):
    """计算在指定数据集上模型的准确率"""
    metric = [0.0, 0.0]  # 正确预测数、总数
    with torch.no_grad():
        for X, y in data_iter:
            metric[0] += accuracy(net(X), y)
            metric[1] += y.numel()
    return metric[0] / metric[1]

### 3.7 训练模型

In [ ]:
def train_epoch(net, train_iter, loss, updater):
    """训练一个迭代周期"""
    metric = [0.0, 0.0, 0.0]  # 训练损失总和、训练准确度总和、样本数
    
    for X, y in train_iter:
        # 计算梯度并更新参数
        y_hat = net(X)
        l = loss(y_hat, y)
        
        if isinstance(updater, torch.optim.Optimizer):
            # 使用PyTorch内置的优化器
            updater.zero_grad()
            l.mean().backward()
            updater.step()
        else:
            # 使用定制的优化器
            l.sum().backward()
            updater(X.shape[0])
        
        metric[0] += float(l.sum())
        metric[1] += float(accuracy(y_hat, y))
        metric[2] += y.numel()
    
    # 返回训练损失和训练准确率
    return metric[0] / metric[2], metric[1] / metric[2]

In [ ]:
def updater(batch_size):
    """SGD优化器"""
    return torch.optim.SGD([W, b], lr=0.1)

# 或者手动实现SGD
lr = 0.1

def sgd_updater(batch_size):
    """小批量随机梯度下降"""
    with torch.no_grad():
        for param in [W, b]:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

In [ ]:
def train(net, train_iter, test_iter, loss, num_epochs, updater):
    """训练模型"""
    train_losses, train_accs, test_accs = [], [], []
    
    for epoch in range(num_epochs):
        train_metrics = train_epoch(net, train_iter, loss, updater)
        test_acc = evaluate_accuracy(net, test_iter)
        
        train_losses.append(train_metrics[0])
        train_accs.append(train_metrics[1])
        test_accs.append(test_acc)
        
        print(f'epoch {epoch + 1}, '
              f'loss {train_metrics[0]:.3f}, '
              f'train acc {train_metrics[1]:.3f}, '
              f'test acc {test_acc:.3f}')
    
    return train_losses, train_accs, test_accs

In [ ]:
# 开始训练
num_epochs = 10
train_losses, train_accs, test_accs = train(
    net, train_iter, test_iter, cross_entropy, num_epochs, sgd_updater)

**可视化训练过程**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 损失曲线
ax1.plot(range(1, num_epochs + 1), train_losses, marker='o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('训练损失')
ax1.grid(True, alpha=0.3)

# 准确率曲线
ax2.plot(range(1, num_epochs + 1), train_accs, marker='o', label='训练集')
ax2.plot(range(1, num_epochs + 1), test_accs, marker='s', label='测试集')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('准确率')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.8 预测

In [ ]:
def predict(net, test_iter, n=6):
    """预测标签"""
    for X, y in test_iter:
        break
    trues = get_fashion_mnist_labels(y)
    preds = get_fashion_mnist_labels(net(X).argmax(axis=1))
    titles = [f'真实: {true}\n预测: {pred}' 
              for true, pred in zip(trues, preds)]
    show_images(X[0:n], 1, n, titles=titles[0:n])
    plt.show()

predict(net, test_iter)

---

## 第四部分: 使用PyTorch的简洁实现

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

### 4.1 定义模型

In [ ]:
# PyTorch不会隐式地调整输入的形状,所以需要flatten
net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 10)
)

# 初始化权重
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)

net.apply(init_weights)

print('模型结构:')
print(net)

### 4.2 定义损失函数

PyTorch的`CrossEntropyLoss`结合了Softmax和交叉熵,数值更稳定。

In [ ]:
loss = nn.CrossEntropyLoss(reduction='none')
print('损失函数:', loss)

### 4.3 定义优化算法

In [ ]:
trainer = torch.optim.SGD(net.parameters(), lr=0.1)
print('优化器:', trainer)

### 4.4 训练模型

In [ ]:
num_epochs = 10
train_losses_c, train_accs_c, test_accs_c = train(
    net, train_iter, test_iter, loss, num_epochs, trainer)

**可视化对比**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 损失对比
ax1.plot(range(1, num_epochs + 1), train_losses, marker='o', label='从零实现')
ax1.plot(range(1, num_epochs + 1), train_losses_c, marker='s', label='简洁实现')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('训练损失对比')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 测试准确率对比
ax2.plot(range(1, num_epochs + 1), test_accs, marker='o', label='从零实现')
ax2.plot(range(1, num_epochs + 1), test_accs_c, marker='s', label='简洁实现')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('测试准确率对比')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.5 预测

In [ ]:
predict(net, test_iter)

---

## 小结

### 核心概念

1. **分类问题**: 预测离散的类别标签
2. **Softmax函数**: 将实数转换为概率分布
   $$\text{softmax}(\mathbf{x})_i = \frac{\exp(x_i)}{\sum_j \exp(x_j)}$$
3. **交叉熵损失**: 衡量两个概率分布的差异
   $$H(y, \hat{y}) = -\sum_i y_i \log \hat{y}_i$$
4. **独热编码**: 将类别表示为向量

### Fashion-MNIST数据集

- 10个服饰类别
- 60,000训练样本 + 10,000测试样本
- 28×28灰度图像

### 从零实现 vs 简洁实现

| 组件 | 从零实现 | 简洁实现 |
|------|----------|----------|
| 模型 | 手动matmul | `nn.Linear` |
| Softmax | 手动实现 | 内置在`CrossEntropyLoss` |
| 损失函数 | 手动实现 | `nn.CrossEntropyLoss` |
| 优化器 | 手动SGD | `torch.optim.SGD` |

### 性能指标

- **训练准确率**: ~85%
- **测试准确率**: ~83%
- 简单的线性模型在Fashion-MNIST上能达到不错的效果!

## 练习

1. **修改批量大小**: 尝试不同的batch_size(32, 128, 512),观察训练速度和准确率
2. **调整学习率**: 测试不同的学习率(0.01, 0.05, 0.2),找到最优值
3. **增加训练轮数**: 训练20个epoch,观察是否过拟合
4. **混淆矩阵**: 实现混淆矩阵,分析哪些类别容易混淆
5. **数据增强**: 尝试随机旋转、翻转等数据增强技术
6. **其他优化器**: 使用Adam、RMSprop等优化器并比较效果
7. **权重可视化**: 可视化学到的权重矩阵W,看看模型学到了什么
8. **迁移到CIFAR-10**: 将代码应用到CIFAR-10数据集(彩色图像)